# Step 5: Adam

Bu notebook'ta Step 4'te oluşturduğum Transformer modelini Adam optimizer ile eğiteceğim.
Model ve veri kümesi aynı kalacak; bu bölümde eğitim sırasında parametrelerin nasıl güncellendiği değişecek.

In [ ]:
import random
import math

random.seed(42)

with open("../data/raw_names.txt", encoding="utf-8") as f:
    docs = [
        line.strip()
        .replace("İ", "i")
        .replace("I", "ı")
        .lower()
        .replace(" ", "")
        for line in f
        if line.strip()
    ]

random.shuffle(docs)

uchars = sorted(set("".join(docs)))
stoi = {ch: i for i, ch in enumerate(uchars)}

BOS = len(uchars)
vocab_size = len(uchars) + 1

print("İsim sayısı:", len(docs))
print("Karakter sayısı:", len(uchars))
print("Sözlük boyutu:", vocab_size)
print("BOS tokenı:", BOS)

In [ ]:
class Value:
    __slots__ = ("data", "grad", "_children", "_local_grads")

    def __init__(self, data, children=(), local_grads=()):
        self.data = float(data)
        self.grad = 0.0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(
            self.data + other.data,
            (self, other),
            (1.0, 1.0),
        )

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(
            self.data * other.data,
            (self, other),
            (other.data, self.data),
        )

    def __pow__(self, power):
        return Value(
            self.data ** power,
            (self,),
            (power * self.data ** (power - 1),),
        )

    def log(self):
        return Value(
            math.log(self.data),
            (self,),
            (1.0 / self.data,),
        )

    def exp(self):
        result = math.exp(self.data)
        return Value(
            result,
            (self,),
            (result,),
        )

    def relu(self):
        return Value(
            max(0.0, self.data),
            (self,),
            (float(self.data > 0),),
        )

    def __neg__(self):
        return self * -1

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other ** -1

    def __rtruediv__(self, other):
        return other * self ** -1

    def backward(self):
        topo = []
        visited = set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)

                for child in node._children:
                    build_topo(child)

                topo.append(node)

        build_topo(self)
        self.grad = 1.0

        for node in reversed(topo):
            for child, local_grad in zip(
                node._children,
                node._local_grads
            ):
                child.grad += local_grad * node.grad

In [ ]:
def linear(x, w):
    return [
        sum(weight * value for weight, value in zip(row, x))
        for row in w
    ]


def softmax(logits):
    max_val = max(value.data for value in logits)
    exps = [(value - max_val).exp() for value in logits]
    total = sum(exps)
    return [value / total for value in exps]


def rmsnorm(x):
    mean_square = sum(value * value for value in x) / len(x)
    scale = (mean_square + 1e-5) ** -0.5
    return [value * scale for value in x]

In [ ]:
n_embd = 16
block_size = 16
n_head = 4
n_layer = 1

assert n_embd % n_head == 0
head_dim = n_embd // n_head

matrix = lambda nout, nin, std=0.08: [
    [Value(random.gauss(0, std)) for _ in range(nin)]
    for _ in range(nout)
]

state_dict = {
    "wte": matrix(vocab_size, n_embd),
    "wpe": matrix(block_size, n_embd),
    "lm_head": matrix(vocab_size, n_embd),
}

for i in range(n_layer):
    state_dict[f"layer{i}.attn_wq"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.attn_wk"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.attn_wv"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.attn_wo"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.mlp_fc1"] = matrix(4 * n_embd, n_embd)
    state_dict[f"layer{i}.mlp_fc2"] = matrix(n_embd, 4 * n_embd)

params = [
    parameter
    for matrix_values in state_dict.values()
    for row in matrix_values
    for parameter in row
]

print("Attention başlığı sayısı:", n_head)
print("Başlık başına boyut:", head_dim)
print("Katman sayısı:", n_layer)
print("Parametre sayısı:", len(params))